# Method Assets

Upload/download precomputed method assets for MEMIT and AlphaEdit. Benchmark datasets are loaded directly from HuggingFace in `edit.ipynb`, so this notebook intentionally does not manage dataset artifacts.


In [ ]:
import shutil
from pathlib import Path

from dotenv import load_dotenv
from huggingface_hub import HfApi, hf_hub_download


load_dotenv()


def find_project_root(start: Path | None = None) -> Path:
    start = (start or Path.cwd()).resolve()
    for path in (start, *start.parents):
        if (path / "src").exists() and (path / "notebook").exists():
            return path
    raise RuntimeError(f"Could not find project root from {start}")


PROJECT_ROOT = find_project_root()
REPO_ID = "Savassan/mnemo"
REMOTE_BASE = "artifacts/qwen2.5-7b"

PROJECTORS_DIR = PROJECT_ROOT / "data" / "alpha_edit_projectors"
LAYER_STATS_DIR = PROJECT_ROOT / "data" / "layer_stats"
ALPHA_EDIT_PROJECTOR = PROJECTORS_DIR / "null_space_project.pt"

ASSET_ROOTS = (
    PROJECTORS_DIR,
    LAYER_STATS_DIR,
)


In [ ]:
def iter_method_asset_files() -> list[Path]:
    files: list[Path] = []

    if ALPHA_EDIT_PROJECTOR.exists():
        files.append(ALPHA_EDIT_PROJECTOR)

    if LAYER_STATS_DIR.exists():
        files.extend(sorted(LAYER_STATS_DIR.rglob("*.npz")))

    return sorted(set(files))


def remote_asset_path(local_file: Path) -> str:
    relative_path = local_file.relative_to(PROJECT_ROOT).as_posix()
    return f"{REMOTE_BASE}/{relative_path}"


def local_asset_path(remote_file: str) -> Path:
    relative_path = remote_file.removeprefix(f"{REMOTE_BASE}/")
    return PROJECT_ROOT / Path(relative_path)


def list_remote_method_assets() -> list[str]:
    api = HfApi()
    prefixes = tuple(
        f"{REMOTE_BASE}/{asset_root.relative_to(PROJECT_ROOT).as_posix()}/"
        for asset_root in ASSET_ROOTS
    )
    return sorted(
        path
        for path in api.list_repo_files(repo_id=REPO_ID, repo_type="dataset")
        if path.startswith(prefixes)
    )


def upload_method_assets() -> None:
    files_to_upload = iter_method_asset_files()
    if not files_to_upload:
        raise FileNotFoundError(
            "No method assets found. Expected AlphaEdit projector and/or layer_stats .npz files."
        )

    api = HfApi()
    for local_file in files_to_upload:
        relative_path = local_file.relative_to(PROJECT_ROOT).as_posix()
        api.upload_file(
            path_or_fileobj=str(local_file),
            path_in_repo=remote_asset_path(local_file),
            repo_id=REPO_ID,
            repo_type="dataset",
        )
        print(f"Uploaded {relative_path}")


def download_method_assets(overwrite: bool = False) -> None:
    remote_files = list_remote_method_assets()
    if not remote_files:
        raise FileNotFoundError(f"No method assets found under {REPO_ID}/{REMOTE_BASE}")

    for remote_file in remote_files:
        destination = local_asset_path(remote_file)
        relative_path = destination.relative_to(PROJECT_ROOT).as_posix()
        destination.parent.mkdir(parents=True, exist_ok=True)

        if destination.exists() and not overwrite:
            print(f"Skipped {relative_path}")
            continue

        cached_file = hf_hub_download(
            repo_id=REPO_ID,
            repo_type="dataset",
            filename=remote_file,
            force_download=overwrite,
        )
        shutil.copy2(cached_file, destination)
        print(f"Downloaded {relative_path}")


In [ ]:
# Inspect local files that would be uploaded:
# iter_method_asset_files()

# Inspect remote files available for download:
# list_remote_method_assets()

# Upload method assets from the local machine:
# upload_method_assets()

# Download method assets on a rented machine:
# download_method_assets()

# Re-download and replace local method assets if needed:
# download_method_assets(overwrite=True)
